In [ ]:
import pandas as pd
import tensorflow as tf
from dataset import Dataset
from model import ElasticityModel
from model_utils import expand_elasticity_matrix_to_original_skus

In [ ]:
sellout_df = pd.read_csv('../data/Sellout_Train.csv')
ihs_df = pd.read_csv('../data/9.18.25_IHS Data_trimmed_v0.1.csv')

In [ ]:
%%time
data = Dataset(sellout_df, ihs_df, 18)

In [ ]:
data_dict = data.process_data()
# data_dict

In [ ]:
data_dict.keys()

In [ ]:
train_data = {
    'log_prices': tf.constant(data_dict['log_prices'], dtype=tf.float32),
    'features': tf.constant(data_dict['features'], dtype=tf.float32),
    'log_volumes': tf.constant(data_dict['log_volumes'], dtype=tf.float32)
}

In [ ]:
 model = ElasticityModel(
        n_skus=len(data_dict['effective_skus']), 
        n_features=data_dict['n_features'], 
        learning_rate=0.0001,
        reg_lambda=0.01
    )
    
history = model.train(train_data, epochs=12000)

In [ ]:
effective_elasticity_matrix = model.get_elasticity_matrix()
effective_elasticity_matrix

In [ ]:
data.sku_mapping

In [ ]:
original_elasticity_matrix = expand_elasticity_matrix_to_original_skus(
        effective_elasticity_matrix, data
    )
original_elasticity_matrix

In [ ]:
print("Number of SKUs not adhering to the own price elasticity condition:",
    (original_elasticity_matrix[original_elasticity_matrix['elasticity_type']=='own_price']['elasticity']>0).sum()
)

print("Number of SKUs not adhering to the cross price elasticity condition:",
    (original_elasticity_matrix[original_elasticity_matrix['elasticity_type']=='cross_price']['elasticity']<0).sum()
     )